# Stage 2 retrain — add the `Not_acne` sixth class

Executes `docs/STAGE2_NEGATIVES_DESIGN.md` end to end: harvest negatives (FFHQ + off-lesion ACNE04),
retrain the 6-class EfficientNetB0, run the three acceptance checks.

**Before running:**
1. `Runtime → Change runtime type → T4 GPU` (A100/L4 roughly halves training time).
2. `skinscan_colab_data.zip` uploaded to the **root of your Google Drive** (MyDrive).

**Total wall time on a T4: ~2.5–3.5 h** (FFHQ download ~4 GB ≈ 10 min, harvest ≈ 25 min, training 1.5–2.5 h, evals ≈ 15 min).

**Safe to Run-all repeatedly.** Every expensive cell checks whether its output already exists (in the session or on Drive)
and skips finished work: clone, unzip, FFHQ shards, harvest, and training are all guarded; harvested crops and the trained
model persist to Drive, so after a disconnect a fresh Run-all fast-forwards to wherever you left off. Training itself
restarts from epoch 0 if interrupted (the trainer has no resume flag), but its best checkpoint is kept on Drive.
The eval cells (§5–§6) always re-run — they are cheap and must reflect the current model.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # no-op if already mounted

## 1. Code + data
The `Not_acne` work (harvester CLI, 6-class vocab) lives on branch **`worktree-issue-2-foundation`**, not `main`.

In [ ]:
![ -d /content/skinscan/.git ] && echo 'repo already cloned' || git clone --depth 1 --branch worktree-issue-2-foundation https://github.com/Kumario1/skinscan.git /content/skinscan
%cd /content/skinscan
!pip install -q ultralytics  # everything else in requirements.txt ships with Colab; fast no-op when installed

In [ ]:
# Detector weights + AcneDataset + ACNE04 + the README §4 test photo.
# The test photo is the LAST file in the zip, so its presence proves the unzip finished.
![ -f data/self_collected/acne-before-scaled-e1764168292784.png ] && echo 'data bundle already extracted' || unzip -q /content/drive/MyDrive/skinscan_colab_data.zip -d /content/skinscan
!mkdir -p /content/drive/MyDrive/skinscan_out
# Resume: crops harvested in a previous session were stashed on Drive (~15 MB) — restore them and their done-markers
![ -f /content/drive/MyDrive/skinscan_out/not_acne_crops.zip ] && unzip -qo /content/drive/MyDrive/skinscan_out/not_acne_crops.zip -d /content/skinscan && echo 'restored harvested crops from a previous session' || echo 'no saved crops on Drive yet — will harvest fresh'
!ls models/detection data/self_collected

## 2. FFHQ faces (1024 px)
Source: [gaunernst/ffhq-1024-wds](https://huggingface.co/datasets/gaunernst/ffhq-1024-wds) — 1000 images per shard.
Shards `00000/01000/02000` → **harvest pool** (3000 faces); shard `03000` → **holdout** for the reject-rate eval.
The design doc requires the holdout sheet to never be seen during harvesting — keep these directories separate.

Each shard leaves a `.done` marker after full extraction, so re-running skips finished shards. If the harvest
itself is already done (marker restored from Drive), the harvest-pool shards are not downloaded at all —
only the holdout shard, which the eval needs every session.

In [ ]:
import io, tarfile
from pathlib import Path
from PIL import Image
from huggingface_hub import hf_hub_download

def pull_shard(shard, out_dir):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    done = out / f'.{shard}.done'
    if done.exists():
        print(shard, 'already extracted ->', out)
        return
    tar_path = hf_hub_download('gaunernst/ffhq-1024-wds', shard, repo_type='dataset')
    n = 0
    with tarfile.open(tar_path) as tf_:
        for m in tf_:
            if not m.name.lower().endswith(('.webp', '.png', '.jpg')):
                continue
            img = Image.open(io.BytesIO(tf_.extractfile(m).read())).convert('RGB')
            img.save(out / (Path(m.name).stem + '.jpg'), quality=95)
            n += 1
    done.touch()
    print(shard, '->', n, 'images in', out)

if Path('data/raw/typeclassification/AcneDataset/.harvest_ffhq.done').exists():
    print('ffhq harvest already done — skipping the harvest-pool shards')
else:
    for shard in ['00000.tar', '01000.tar', '02000.tar']:
        pull_shard(shard, 'data/raw/ffhq')          # harvest pool
pull_shard('03000.tar', 'data/raw/ffhq_holdout')      # eval only — NEVER harvested

## 3. Harvest `Not_acne` negatives
Sizing (resolves the design doc's OPEN item, from the live per-class totals train+valid+test):
Blackheads 1240 · Cyst 1040 · Papules 1032 · Pustules 1006 · Whiteheads 299 → median ≈ 1032, largest 1240.
Target **Not_acne ≈ 1000 total** (650 FFHQ + 350 ACNE04 off-lesion) — near the median, under the largest.
The harvester splits 60/20/20 itself and defaults to the locked operating point (conf 0.07 / IoU 0.2 / imgsz 1024).

Each source is guarded by a done-marker and the finished crops are stashed to Drive. To **force a re-harvest**
(e.g. with different limits): delete `data/raw/typeclassification/AcneDataset/.harvest_*.done` here and
`skinscan_out/not_acne_crops.zip` on Drive, then re-run from the FFHQ cell.

In [ ]:
!python -m src.classification.train_type_classifier --inspect  # before harvest (5 classes on a fresh session)

In [ ]:
![ -f data/raw/typeclassification/AcneDataset/.harvest_ffhq.done ] && echo 'ffhq harvest already done' || (python -m src.classification.harvest_negatives --mode ffhq --images data/raw/ffhq --limit 650 && touch data/raw/typeclassification/AcneDataset/.harvest_ffhq.done)
![ -f data/raw/typeclassification/AcneDataset/.harvest_acne04.done ] && echo 'acne04 harvest already done' || (python -m src.classification.harvest_negatives --mode acne04 --limit 350 && touch data/raw/typeclassification/AcneDataset/.harvest_acne04.done)
# Stash crops + markers on Drive so a disconnect never costs a re-harvest (-FS keeps the zip in sync)
!zip -q -FS -r /content/drive/MyDrive/skinscan_out/not_acne_crops.zip data/raw/typeclassification/AcneDataset/train/Not_acne data/raw/typeclassification/AcneDataset/valid/Not_acne data/raw/typeclassification/AcneDataset/test/Not_acne data/raw/typeclassification/AcneDataset/.harvest_ffhq.done data/raw/typeclassification/AcneDataset/.harvest_acne04.done
!python -m src.classification.train_type_classifier --inspect  # after: Not_acne beside the 5 real classes

Check the second `--inspect`: `Not_acne` train count must stay ≤ 735 (largest real train class, Blackheads).
If the FFHQ harvest printed fewer than 650 crops (low detector yield on clean skin): delete the marker, add a shard,
and re-run the harvest cell — re-processing the same images overwrites the same files, no duplicates:
```python
!rm data/raw/typeclassification/AcneDataset/.harvest_ffhq.done
pull_shard('04000.tar', 'data/raw/ffhq')
```

## 4. Retrain (150 epochs, ~1.5–2.5 h on T4)
`--out` points at Drive so the best checkpoint survives a disconnect. The class-order guard passes because
`RAW_ACNE_CLASSES` on this branch is already the 6-list with `Not_acne` at index 2.

The labels JSON is written only **after** training completes, so its presence on Drive means a finished run —
the cell then skips. To **force a retrain**, delete `skinscan_out/acne_model.keras` and
`skinscan_out/acne_model.keras.labels.json` on Drive.

**Acceptance check #2 (no type regression):** in the printed classification report, the mean F1 of the **5 original
classes** (ignore the `Not_acne` row) must be **≥ 0.90** (baseline macro-F1 0.92, ≤ 2 pt drop allowed).

In [ ]:
![ -f /content/drive/MyDrive/skinscan_out/acne_model.keras.labels.json ] && echo 'trained model already on Drive — skipping (delete skinscan_out/acne_model.keras* on Drive to retrain)' || python -m src.classification.train_type_classifier --out /content/drive/MyDrive/skinscan_out/acne_model.keras

In [ ]:
# Install the new model where the pipeline's config defaults expect it (cheap — always refresh from Drive)
!mkdir -p models/classification  # gitignored, so neither the clone nor the data bundle creates it
!cp /content/drive/MyDrive/skinscan_out/acne_model.keras models/classification/acne_model.keras
!cp /content/drive/MyDrive/skinscan_out/acne_model.keras.labels.json models/classification/acne_model.keras.labels.json

## 5. Acceptance check #1 — FFHQ holdout reject rate ≥ 80%
Detector boxes on never-harvested clean faces are false positives by construction; ≥ 80% must classify `Not_acne`.
Always re-runs (≈ 10 min) so the numbers always match the model installed above.

In [ ]:
!python -m src.classification.run_acne04_pipeline --images data/raw/ffhq_holdout --limit 500 --max-boxes 100 --out runs/ffhq_reject

In [ ]:
import json, pathlib
recs = json.loads(pathlib.Path('runs/ffhq_reject/predictions.json').read_text())
dets = [d for r in recs for d in r['detections']]
na = sum(d['prediction'] == 'Not_acne' for d in dets)
print(f"{na}/{len(dets)} = {na/max(len(dets),1):.1%} rejected as Not_acne (need >= 80%)")

## 6. Acceptance check #3 — phantom-pustules re-run
Same README §4 image that produced `Pustules=16`; several weak boxes (detector conf 0.19–0.37) should flip to `Not_acne`.

In [ ]:
!python -m src.classification.run_acne04_pipeline --image data/self_collected/acne-before-scaled-e1764168292784.png --max-boxes 100 --out runs/phantom_recheck
import json
r = json.load(open('runs/phantom_recheck/predictions.json'))[0]
print('was: Pustules=16   now:', r['acne_type_counts'])

In [ ]:
!python -m pytest -q

## 7. Export eval artifacts to Drive

In [ ]:
!zip -q -FS -r /content/drive/MyDrive/skinscan_out/eval_runs.zip runs/ffhq_reject runs/phantom_recheck
!ls -lh /content/drive/MyDrive/skinscan_out

## Done — back on the Mac
Download from Drive `skinscan_out/`:
- `acne_model.keras` + `acne_model.keras.labels.json` → replace the two files in `models/classification/` (gitignored, nothing to commit)
- `eval_runs.zip` → new counts + crop sheets for the README §4 update, committed on `worktree-issue-2-foundation`